In [ ]:
mssparkutils.notebook.run("config", 60)

In [ ]:
empresas = [
    {
    "nome": "empresa 01", 
    "app_key": spark.conf.get("empresa_01_app_key"),
    "app_secret": spark.conf.get("empresa_01_app_secret")
    },
    {
    "nome": "empresa 02",
    "app_key": spark.conf.get("empresa_02_app_key"),
    "app_secret": spark.conf.get("empresa_02_app_secret")
    },
    {
    "nome": "empresa 03",
    "app_key": spark.conf.get("empresa_03_app_key"),
    "app_secret": spark.conf.get("empresa_03_app_secret")
    }
]

In [ ]:
import requests

url = "https://app.omie.com.br/api/v1/geral/contacorrente/"
call = "ListarContasCorrentes"
data_key = "ListarContasCorrentes"
nome_tabela = "contas"

In [ ]:
def get_dados(call, app_key, app_secret, data_key):
    pagina = 1 
    todos_dados = []

    while True: 
        payload = { 
            "call": call,
            "app_key": app_key,
            "app_secret": app_secret,
            "param": [{
                "pagina": pagina,
                "registros_por_pagina": 50
            }]
        }

        response = requests.post(url, json=payload)

        if response.status_code != 200:
            raise Exception(f"Erro na API: {response.text}")

        data = response.json()

        dados = data.get(data_key, [])

        if not dados:
            break

        todos_dados.extend(dados)
        if pagina >= data.get("total_de_paginas", 1):
            break

        pagina += 1

    return todos_dados

In [ ]:
todos_dados = []
for emp in empresas: 
    print(f"Processando {emp['nome']}...")

    tabela = get_dados(call, emp["app_key"], emp["app_secret"], data_key)

    for l in tabela:
        l["empresa"] = emp["nome"]
    
    todos_dados.extend(tabela)

In [ ]:
import json

df_bronze = spark.createDataFrame(
    [(json.dumps(d),) for d in todos_dados],
    ["raw_json"]
)

In [ ]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp

df_bronze = df_bronze.withColumn("ingestion_ts", from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))
display(df_bronze)

In [ ]:
caminho = f"Files/bronze/{nome_tabela}"
df_bronze.write.format("delta").mode("append").save(caminho)
print(f"Tabela {nome_tabela} carregada com sucesso")